In [1]:
%cd ../..

/Users/katyscott/Documents/BHKLab_GitHub/recist-vs-reality


In [2]:
from damply import dirs
from pathlib import Path
import pandas as pd
import SimpleITK as sitk
import numpy as np
from imgtools.transforms.functional import resample

/Users/katyscott/Documents/BHKLab_GitHub/recist-vs-reality/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
from skimage.measure import label

def load_img(path,
             desired_spacing=[1.0,1.0,1.0]
             ) -> tuple[np.array, int]:

    img = sitk.ReadImage(path)
    img_spacing = list(img.GetSpacing())

    if img_spacing != desired_spacing:
        img = resample(img, spacing = desired_spacing, interpolation='nearest')

    img_arr = sitk.GetArrayFromImage(img)

    if np.count_nonzero(img_arr) > 0: 
        conn_comps, num_comps = label(img_arr, return_num=True)

    return conn_comps, num_comps

In [21]:
def get_max_slice(np_mask:np.array) -> tuple[np.array, int]:
    # Sum the mask in the x and y axes to find the axial slice with the largest tumour area
    axial_sum = np.sum(np_mask, axis=(1,2))
    # Get the index of the axial slice with the largest tumour area
    max_axial_index = np.argmax(axial_sum)
    # Select out the slice with the largest index
    max_area_slice = np_mask[max_axial_index]

    return max_area_slice, max_axial_index


def bbox_from_seg(seg_2d: np.ndarray) -> list[np.int64]: 
    '''  
    Get a bounding box from a given segmentation (binary array). Assumes that only one tumour is within this segmentation.

    Parameters
    ----------
    seg2d: np.ndarray
        The segmentation slice to get a bounding box from 

    Returns
    ----------
    bbox: list 
        A list of coordinates for the bounding box in the order of [xmin, ymin, xmax, ymax]
    '''
    rows = np.any(seg_2d, axis=1)
    cols = np.any(seg_2d, axis=0)
    ymin, ymax = np.where(rows)[0][[0, -1]]
    xmin, xmax = np.where(cols)[0][[0, -1]]

    bbox = [xmin, ymin, xmax, ymax]
    
    return bbox


def proc_component(comps:np.ndarray,
                   comp_idx:int,
                   thresh_inc:int = 10
                   ) -> tuple[np.array, int] | tuple[int, int]:
    # Set all other components to 0s
    curr_comp = np.ma.masked_where(comps != comp_idx, comps).filled(0)

    # check if size of component is large enough
    if np.count_nonzero(curr_comp) < thresh_inc:
            return -1, -1
      
    curr_comp_bin = np.clip(curr_comp, 0, 1)
    curr_comp_bin = curr_comp_bin.astype(int)

    max_axial_slice, max_axial_index = get_max_slice(curr_comp_bin)

    return curr_comp_bin, max_axial_index

In [22]:
def calc_2d_IoU(bbox_gt: np.ndarray,
                bbox_pred: np.ndarray
                ) -> np.float64:
    '''  
    Calculate the 2D intersection-over-union (IoU) between two bounding boxes. 

    Parameters
    ----------
    bbox_gt: np.ndarray
        Ground truth 2D bounding box in the order [xmin, ymin, xmax, ymax]
    bbox_pred: np.ndarray
        Predicted 2D bounding box in the order [xmin, ymin, xmax, ymax]

    Returns 
    ----------
    iou: float 
        The IoU of the two bounding boxes
    '''
    # Get coordinates and area of intersecting rectangle
    xmin_inter = max(bbox_gt[0], bbox_pred[0])
    ymin_inter = max(bbox_gt[1], bbox_pred[1])
    xmax_inter = min(bbox_gt[2], bbox_pred[2])
    ymax_inter = min(bbox_gt[3], bbox_pred[3])

    inter_width = max(0, xmax_inter-xmin_inter)
    inter_height = max(0, ymax_inter-ymin_inter)
    inter_area = inter_width * inter_height 

    # Get area of the each bounding box 
    gt_area = (bbox_gt[2]-bbox_gt[0]) * (bbox_gt[3]-bbox_gt[1])
    pred_area = (bbox_pred[2]-bbox_pred[0]) * (bbox_pred[3]-bbox_pred[1])

    # Calculate union area 
    union_area = gt_area + pred_area - inter_area 

    # Calculate IoU 
    if union_area == 0: 
        return 0.0 # This should never happen, but implementing just in case
    
    iou = inter_area / union_area 

    return iou

In [12]:
dataset = "PASTA"
gt_folder = dirs.RAWDATA / dataset / "gt"
pred_folder = dirs.PROCDATA / dataset / "pred"

gt_files = list(gt_folder.glob("*.nii.gz"))
pred_files = list(pred_folder.glob("*.nii.gz"))

# Get set of samples for each
gt_samples = {file.name for file in gt_files}
pred_samples = {file.name for file in pred_files}

samples_to_proc = pred_samples & gt_samples

In [26]:
# sample_id = "LesionLocator_1954.nii.gz"
thresh_inc = 10
match_dict = {}


for sample_id in samples_to_proc:
    gt_path = gt_folder / sample_id
    pred_path = pred_folder / sample_id

    gt_conn_comps, gt_num_comps = load_img(gt_path)
    pred_conn_comps, pred_num_comps = load_img(pred_path)

    # Something about the resampling means that ground truth has to get flipped back to normal
    gt_conn_comps = np.flip(gt_conn_comps, axis=[1,2])


    for gt_idx in range(1, gt_num_comps+1):
        curr_gt_comp, gt_max_axial_index = proc_component(gt_conn_comps, gt_idx, thresh_inc)

        if isinstance(curr_gt_comp, int) and curr_gt_comp -1:
            print("too small")
            continue

        # Check if there are any overlaps with the individual predicted segmentations 
        
        match_found = False
        for pred_idx in range(1,pred_num_comps+1): 
            curr_pred_comp, pred_max_axial_index = proc_component(pred_conn_comps, pred_idx, thresh_inc)

            if isinstance(curr_pred_comp, int) and curr_pred_comp -1:
                print("too small")
                continue

            curr_intersect = curr_gt_comp & curr_pred_comp
            if np.count_nonzero(curr_intersect) > 0: 
                # Indicates overlap between the two segmentations, so these have matched. Save as a pair for further analysis
                print('match found!')

                gt_bbox = bbox_from_seg(curr_gt_comp[gt_max_axial_index])
                pred_bbox = bbox_from_seg(curr_pred_comp[gt_max_axial_index])

                match_2D_iou = calc_2d_IoU(gt_bbox, pred_bbox)

                match_id = f"{sample_id.removesuffix(".nii.gz")}_gt{gt_idx}_pred{pred_idx}"
                match_dict[match_id] = {
                    "sample_id": sample_id,
                    "gt_bbox": list(map(int, gt_bbox)),
                    "pred_bbox": list(map(int, pred_bbox)),
                    "slice_idx_from_gt": int(gt_max_axial_index),
                    "slice_2D_iou": float(match_2D_iou) 
                }



match found!
match found!
match found!
match found!
too small
too small
too small
too small
too small
too small


In [27]:
match_dict

{'LesionLocator_1954_gt1_pred1': {'sample_id': 'LesionLocator_1954.nii.gz',
  'gt_bbox': [153, 265, 167, 280],
  'pred_bbox': [156, 267, 162, 275],
  'slice_idx_from_gt': 123,
  'slice_2D_iou': 0.22857142857142856},
 'LesionLocator_1954_gt2_pred2': {'sample_id': 'LesionLocator_1954.nii.gz',
  'gt_bbox': [121, 303, 149, 327],
  'pred_bbox': [119, 301, 141, 326],
  'slice_idx_from_gt': 153,
  'slice_2D_iou': 0.6036745406824147},
 'LesionLocator_1954_gt3_pred3': {'sample_id': 'LesionLocator_1954.nii.gz',
  'gt_bbox': [172, 282, 188, 300],
  'pred_bbox': [174, 284, 186, 296],
  'slice_idx_from_gt': 153,
  'slice_2D_iou': 0.5},
 'LesionLocator_0001_gt1_pred1': {'sample_id': 'LesionLocator_0001.nii.gz',
  'gt_bbox': [148, 283, 173, 308],
  'pred_bbox': [153, 287, 171, 306],
  'slice_idx_from_gt': 149,
  'slice_2D_iou': 0.5472}}

In [28]:
pd.DataFrame().from_dict(match_dict, orient='index')

,sample_id,gt_bbox,pred_bbox,slice_idx_from_gt,slice_2D_iou
LesionLocator_1954_gt1_pred1,LesionLocator_1954.nii.gz,"[153, 265, 167, 280]","[156, 267, 162, 275]",123,0.228571
LesionLocator_1954_gt2_pred2,LesionLocator_1954.nii.gz,"[121, 303, 149, 327]","[119, 301, 141, 326]",153,0.603675
LesionLocator_1954_gt3_pred3,LesionLocator_1954.nii.gz,"[172, 282, 188, 300]","[174, 284, 186, 296]",153,0.500000
LesionLocator_0001_gt1_pred1,LesionLocator_0001.nii.gz,"[148, 283, 173, 308]","[153, 287, 171, 306]",149,0.547200


In [ ]:
print(gt_max_axial_index)
print(pred_max_axial_index)

In [ ]:
gt_comp_img = sitk.GetImageFromArray(gt_conn_comps)
sitk.WriteImage(gt_comp_img, dirs.PROCDATA / dataset / "gt_resample" / sample_id)

In [ ]:
flip_pred = np.flip(pred_conn_comps, 2)

np.nonzero(flip_pred)

In [13]:
path = gt_path
img = sitk.ReadImage(path)
img_spacing = list(img.GetSpacing())

if img_spacing != [1.0, 1.0, 1.0]:
    resampled_img = resample(img, spacing = [1.0, 1.0, 1.0], interpolation='nearest')

In [16]:
img_arr = sitk.GetArrayFromImage(img)
resampled_img_arr = sitk.GetArrayFromImage(resampled_img)

In [17]:
np.nonzero(img_arr)

(array([25, 25, 25, ..., 32, 32, 32], shape=(2863,)),
 array([224, 224, 225, ..., 222, 223, 223], shape=(2863,)),
 array([346, 347, 344, ..., 333, 330, 331], shape=(2863,)))

In [18]:
np.nonzero(resampled_img_arr)

(array([123, 123, 123, ..., 162, 162, 162], shape=(13410,)),
 array([218, 218, 219, ..., 216, 217, 217], shape=(13410,)),
 array([337, 338, 335, ..., 325, 322, 323], shape=(13410,)))